# Integrating diversity  and gisaid_data

In [1]:
# General
import numpy as np
import pandas as pd 
import os
import bs4
import requests
import warnings
import re
import html
import unidecode

In [2]:
df_diversity = pd.read_excel("diversity_df_new.xlsx")

df_diversity

,location,week,weekly_cases,n_seq_perwk,sequenc_eff,richness,diversity
0,Afghanistan,2019-12-29,0,0,0.0,NaN,NaN
1,Afghanistan,2020-01-05,0,0,0.0,NaN,NaN
2,Afghanistan,2020-01-12,0,0,0.0,NaN,NaN
3,Afghanistan,2020-01-19,0,0,0.0,NaN,NaN
4,Afghanistan,2020-01-26,0,0,0.0,NaN,NaN
...,...,...,...,...,...,...,...
33577,Zimbabwe,2023-03-26,41,0,0.0,NaN,NaN
33578,Zimbabwe,2023-04-02,29,0,0.0,NaN,NaN
33579,Zimbabwe,2023-04-09,29,0,0.0,NaN,NaN
33580,Zimbabwe,2023-04-16,25,0,0.0,NaN,NaN


In [3]:
df_diversity = df_diversity.replace({'USA' : 'United States'})

In [7]:
df_gisaid = pd.read_excel("gisaid_data_w_new.xlsx")

df_gisaid

,location,week,pango_lineage,VOC,n,proportion,n_seq_perwk
0,Afghanistan,2019-12-29,ND,ND,0,0.0,0
1,Afghanistan,2020-01-05,ND,ND,0,0.0,0
2,Afghanistan,2020-01-12,ND,ND,0,0.0,0
3,Afghanistan,2020-01-19,ND,ND,0,0.0,0
4,Afghanistan,2020-01-26,ND,ND,0,0.0,0
...,...,...,...,...,...,...,...
484013,Zimbabwe,2023-03-26,ND,ND,0,0.0,0
484014,Zimbabwe,2023-04-02,ND,ND,0,0.0,0
484015,Zimbabwe,2023-04-09,ND,ND,0,0.0,0
484016,Zimbabwe,2023-04-16,ND,ND,0,0.0,0


In [8]:
df_gisaid = df_gisaid.replace({'USA' : 'United States'})

In [9]:
df_gisaid = df_gisaid.groupby(['location', 'week', 'VOC']).sum()
df_gisaid = df_gisaid.reset_index()
df_gisaid

,location,week,VOC,n,proportion,n_seq_perwk
0,Afghanistan,2019-12-29,ND,0,0.0,0
1,Afghanistan,2020-01-05,ND,0,0.0,0
2,Afghanistan,2020-01-12,ND,0,0.0,0
3,Afghanistan,2020-01-19,ND,0,0.0,0
4,Afghanistan,2020-01-26,ND,0,0.0,0
...,...,...,...,...,...,...
72119,Zimbabwe,2023-03-26,ND,0,0.0,0
72120,Zimbabwe,2023-04-02,ND,0,0.0,0
72121,Zimbabwe,2023-04-09,ND,0,0.0,0
72122,Zimbabwe,2023-04-16,ND,0,0.0,0


In [10]:
df_gisaid.to_csv("df_w_voc_gisaid.csv", index=False)

In [11]:
df_gisaid.iloc[32687:32695,:]

,location,week,VOC,n,proportion,n_seq_perwk
32687,Japan,2022-10-23,Omicron BA.2*,57,0.011517,74235
32688,Japan,2022-10-23,Omicron BA.2.75*,117,0.023641,128674
32689,Japan,2022-10-23,Omicron BA.4*,9,0.001819,19796
32690,Japan,2022-10-23,Omicron BA.5*,4540,0.917357,529543
32691,Japan,2022-10-23,Omicron BQ.*,182,0.036775,128674
32692,Japan,2022-10-23,Recombinants,7,0.001414,19796
32693,Japan,2022-10-23,XBB*,36,0.007274,19796
32694,Japan,2022-10-23,other,1,0.000202,4949


In [12]:
df_gisaid = df_gisaid.drop(["n_seq_perwk"], axis=1)
df_gisaid.iloc[32687:32695,:]

,location,week,VOC,n,proportion
32687,Japan,2022-10-23,Omicron BA.2*,57,0.011517
32688,Japan,2022-10-23,Omicron BA.2.75*,117,0.023641
32689,Japan,2022-10-23,Omicron BA.4*,9,0.001819
32690,Japan,2022-10-23,Omicron BA.5*,4540,0.917357
32691,Japan,2022-10-23,Omicron BQ.*,182,0.036775
32692,Japan,2022-10-23,Recombinants,7,0.001414
32693,Japan,2022-10-23,XBB*,36,0.007274
32694,Japan,2022-10-23,other,1,0.000202


In [13]:
df_diversity_exp = df_diversity[["location", "week", "weekly_cases", "sequenc_eff" ,"richness", "diversity"]]

df_diversity_exp

,location,week,weekly_cases,sequenc_eff,richness,diversity
0,Afghanistan,2019-12-29,0,0.0,NaN,NaN
1,Afghanistan,2020-01-05,0,0.0,NaN,NaN
2,Afghanistan,2020-01-12,0,0.0,NaN,NaN
3,Afghanistan,2020-01-19,0,0.0,NaN,NaN
4,Afghanistan,2020-01-26,0,0.0,NaN,NaN
...,...,...,...,...,...,...
33577,Zimbabwe,2023-03-26,41,0.0,NaN,NaN
33578,Zimbabwe,2023-04-02,29,0.0,NaN,NaN
33579,Zimbabwe,2023-04-09,29,0.0,NaN,NaN
33580,Zimbabwe,2023-04-16,25,0.0,NaN,NaN


In [14]:
df_gisaid_exp = df_gisaid.copy()

df_gisaid_exp = df_gisaid_exp.merge(df_diversity_exp, on=["location", "week"], how='left')

df_gisaid_exp

,location,week,VOC,n,proportion,weekly_cases,sequenc_eff,richness,diversity
0,Afghanistan,2019-12-29,ND,0,0.0,0.0,0.0,NaN,NaN
1,Afghanistan,2020-01-05,ND,0,0.0,0.0,0.0,NaN,NaN
2,Afghanistan,2020-01-12,ND,0,0.0,0.0,0.0,NaN,NaN
3,Afghanistan,2020-01-19,ND,0,0.0,0.0,0.0,NaN,NaN
4,Afghanistan,2020-01-26,ND,0,0.0,0.0,0.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...
72119,Zimbabwe,2023-03-26,ND,0,0.0,41.0,0.0,NaN,NaN
72120,Zimbabwe,2023-04-02,ND,0,0.0,29.0,0.0,NaN,NaN
72121,Zimbabwe,2023-04-09,ND,0,0.0,29.0,0.0,NaN,NaN
72122,Zimbabwe,2023-04-16,ND,0,0.0,25.0,0.0,NaN,NaN


In [15]:
df_gisaid_exp.iloc[32687:32695,:]

,location,week,VOC,n,proportion,weekly_cases,sequenc_eff,richness,diversity
32687,Japan,2022-10-23,Omicron BA.2*,57,0.011517,281974.0,0.02,187.0,16.01966
32688,Japan,2022-10-23,Omicron BA.2.75*,117,0.023641,281974.0,0.02,187.0,16.01966
32689,Japan,2022-10-23,Omicron BA.4*,9,0.001819,281974.0,0.02,187.0,16.01966
32690,Japan,2022-10-23,Omicron BA.5*,4540,0.917357,281974.0,0.02,187.0,16.01966
32691,Japan,2022-10-23,Omicron BQ.*,182,0.036775,281974.0,0.02,187.0,16.01966
32692,Japan,2022-10-23,Recombinants,7,0.001414,281974.0,0.02,187.0,16.01966
32693,Japan,2022-10-23,XBB*,36,0.007274,281974.0,0.02,187.0,16.01966
32694,Japan,2022-10-23,other,1,0.000202,281974.0,0.02,187.0,16.01966


In [17]:
df_gisaid_exp.to_csv("diversity_gisaid_w.csv", index=False)

In [18]:
df_gisaid_exp["VOC_case_count"] = df_gisaid_exp["proportion"] * df_gisaid_exp["weekly_cases"]

df_gisaid_exp

,location,week,VOC,n,proportion,weekly_cases,sequenc_eff,richness,diversity,VOC_case_count
0,Afghanistan,2019-12-29,ND,0,0.0,0.0,0.0,NaN,NaN,0.0
1,Afghanistan,2020-01-05,ND,0,0.0,0.0,0.0,NaN,NaN,0.0
2,Afghanistan,2020-01-12,ND,0,0.0,0.0,0.0,NaN,NaN,0.0
3,Afghanistan,2020-01-19,ND,0,0.0,0.0,0.0,NaN,NaN,0.0
4,Afghanistan,2020-01-26,ND,0,0.0,0.0,0.0,NaN,NaN,0.0
...,...,...,...,...,...,...,...,...,...,...
72119,Zimbabwe,2023-03-26,ND,0,0.0,41.0,0.0,NaN,NaN,0.0
72120,Zimbabwe,2023-04-02,ND,0,0.0,29.0,0.0,NaN,NaN,0.0
72121,Zimbabwe,2023-04-09,ND,0,0.0,29.0,0.0,NaN,NaN,0.0
72122,Zimbabwe,2023-04-16,ND,0,0.0,25.0,0.0,NaN,NaN,0.0


In [19]:
df_gisaid_exp.iloc[32687:32695,:]

,location,week,VOC,n,proportion,weekly_cases,sequenc_eff,richness,diversity,VOC_case_count
32687,Japan,2022-10-23,Omicron BA.2*,57,0.011517,281974.0,0.02,187.0,16.01966,3247.629420
32688,Japan,2022-10-23,Omicron BA.2.75*,117,0.023641,281974.0,0.02,187.0,16.01966,6666.186704
32689,Japan,2022-10-23,Omicron BA.4*,9,0.001819,281974.0,0.02,187.0,16.01966,512.783593
32690,Japan,2022-10-23,Omicron BA.5*,4540,0.917357,281974.0,0.02,187.0,16.01966,258670.834512
32691,Japan,2022-10-23,Omicron BQ.*,182,0.036775,281974.0,0.02,187.0,16.01966,10369.623762
32692,Japan,2022-10-23,Recombinants,7,0.001414,281974.0,0.02,187.0,16.01966,398.831683
32693,Japan,2022-10-23,XBB*,36,0.007274,281974.0,0.02,187.0,16.01966,2051.134371
32694,Japan,2022-10-23,other,1,0.000202,281974.0,0.02,187.0,16.01966,56.975955


In [20]:
df_gisaid_exp.columns

Index(['location', 'week', 'VOC', 'n', 'proportion', 'weekly_cases',
       'sequenc_eff', 'richness', 'diversity', 'VOC_case_count'],
      dtype='object')

In [21]:
df_gisaid_exp_f1 = df_gisaid_exp[['location', 'week', 'n']]

df_gisaid_exp_f1

,location,week,n
0,Afghanistan,2019-12-29,0
1,Afghanistan,2020-01-05,0
2,Afghanistan,2020-01-12,0
3,Afghanistan,2020-01-19,0
4,Afghanistan,2020-01-26,0
...,...,...,...
72119,Zimbabwe,2023-03-26,0
72120,Zimbabwe,2023-04-02,0
72121,Zimbabwe,2023-04-09,0
72122,Zimbabwe,2023-04-16,0


In [22]:
df_gisaid_exp_f1 = df_gisaid_exp_f1.groupby(['location', 'week']).sum()
df_gisaid_exp_f1 = df_gisaid_exp_f1.reset_index()

df_gisaid_exp_f1

,location,week,n
0,Afghanistan,2019-12-29,0
1,Afghanistan,2020-01-05,0
2,Afghanistan,2020-01-12,0
3,Afghanistan,2020-01-19,0
4,Afghanistan,2020-01-26,0
...,...,...,...
37057,Zimbabwe,2023-03-26,0
37058,Zimbabwe,2023-04-02,0
37059,Zimbabwe,2023-04-09,0
37060,Zimbabwe,2023-04-16,0


In [23]:
df_gisaid_exp_f1.iloc[16328:16331,:]

,location,week,n
16328,Japan,2022-10-16,4419
16329,Japan,2022-10-23,4949
16330,Japan,2022-10-30,5036


In [24]:
df_gisaid_exp_f2 = df_gisaid_exp[['location', 'week', 'VOC', 'VOC_case_count']]

df_gisaid_exp_f2

,location,week,VOC,VOC_case_count
0,Afghanistan,2019-12-29,ND,0.0
1,Afghanistan,2020-01-05,ND,0.0
2,Afghanistan,2020-01-12,ND,0.0
3,Afghanistan,2020-01-19,ND,0.0
4,Afghanistan,2020-01-26,ND,0.0
...,...,...,...,...
72119,Zimbabwe,2023-03-26,ND,0.0
72120,Zimbabwe,2023-04-02,ND,0.0
72121,Zimbabwe,2023-04-09,ND,0.0
72122,Zimbabwe,2023-04-16,ND,0.0


In [25]:
df_gisaid_exp_f2 = pd.pivot_table(df_gisaid_exp_f2, values = 'VOC_case_count', index=['location','week'], columns = 'VOC').reset_index()

df_gisaid_exp_f2

VOC,location,week,Alpha*,Beta,Delta*,Epsilon,Eta,Gamma,Iota,Kappa,...,Omicron BA.2*,Omicron BA.2.75*,Omicron BA.4*,Omicron BA.5*,Omicron BQ.*,Recombinants,XBB*,other,wildtype A*,wildtype B*
0,Afghanistan,2019-12-29,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Afghanistan,2020-01-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Afghanistan,2020-01-12,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Afghanistan,2020-01-19,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Afghanistan,2020-01-26,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
33577,Zimbabwe,2023-03-26,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
33578,Zimbabwe,2023-04-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
33579,Zimbabwe,2023-04-09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
33580,Zimbabwe,2023-04-16,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [26]:
df_gisaid_exp_f2 = df_gisaid_exp_f2.groupby(['location', 'week']).sum()
df_gisaid_exp_f2 = df_gisaid_exp_f2.reset_index()

df_gisaid_exp_f2

VOC,location,week,Alpha*,Beta,Delta*,Epsilon,Eta,Gamma,Iota,Kappa,...,Omicron BA.2*,Omicron BA.2.75*,Omicron BA.4*,Omicron BA.5*,Omicron BQ.*,Recombinants,XBB*,other,wildtype A*,wildtype B*
0,Afghanistan,2019-12-29,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,Afghanistan,2020-01-05,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,Afghanistan,2020-01-12,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,Afghanistan,2020-01-19,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,Afghanistan,2020-01-26,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
33577,Zimbabwe,2023-03-26,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
33578,Zimbabwe,2023-04-02,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
33579,Zimbabwe,2023-04-09,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
33580,Zimbabwe,2023-04-16,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [27]:
df_gisaid_exp_f2.iloc[14936:14939,:]

VOC,location,week,Alpha*,Beta,Delta*,Epsilon,Eta,Gamma,Iota,Kappa,...,Omicron BA.2*,Omicron BA.2.75*,Omicron BA.4*,Omicron BA.5*,Omicron BQ.*,Recombinants,XBB*,other,wildtype A*,wildtype B*
14936,Japan,2022-10-16,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1956.604209,4759.307536,370.168364,219033.909029,5764.050238,211.524779,1480.673456,0.000000,0.0,52.881195
14937,Japan,2022-10-23,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,3247.629420,6666.186704,512.783593,258670.834512,10369.623762,398.831683,2051.134371,56.975955,0.0,0.000000
14938,Japan,2022-10-30,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,4945.386418,14357.573471,638.114376,360375.094122,18744.609809,478.585782,1994.107427,159.528594,0.0,0.000000


In [28]:
df_gisaid_exp_f1.to_csv("df_gisaid_exp_f1.csv", index=False)

df_gisaid_exp_f2.to_csv("df_gisaid_exp_f2.csv", index=False)

In [31]:
ttt_df = df_gisaid_exp_f1[df_gisaid_exp_f1["location"] == 'United States']
ttt_df

,location,week,n
35322,United States,2019-12-29,1
35323,United States,2020-01-05,0
35324,United States,2020-01-12,1
35325,United States,2020-01-19,13
35326,United States,2020-01-26,9
...,...,...,...
35491,United States,2023-03-26,5129
35492,United States,2023-04-02,4523
35493,United States,2023-04-09,2418
35494,United States,2023-04-16,385
